# ⚡ Small Transformer Inference (CPU Baseline)

This notebook demonstrates how to achieve **high-performance inference** on a CPU using DistilBERT and **Dynamic Quantization**.

**Tasks:**
1. **Sentiment Analysis**: Classifying text as Positive/Negative.
2. **NER**: Extracting entities (Names, Locations) from text.
3. **Optimization**: Quantizing the model to `int8`.

In [ ]:
# 1. Install Dependencies
%pip install torch transformers numpy pandas

import torch
import time
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 🧠 CPU Optimization: Control Threads
# Setting this to the number of physical cores is usually best for latency.
torch.set_num_threads(os.cpu_count())
print(f"✅ Threads set to: {torch.get_num_threads()}")

## 1. Sentiment Analysis Baseline (FP32)
We load `distilbert-base-uncased-finetuned-sst-2-english`. It is a standard baseline for sentiment.

In [4]:
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Load Model & Tokenizer
print(f"⬇️  Downloading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp32 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Save model locally so we can accurately measure its size
model_fp32.save_pretrained("./model_fp32")
tokenizer.save_pretrained("./model_fp32")

# FIX: Check for either .bin (standard) or .safetensors (newer default)
if os.path.exists("./model_fp32/pytorch_model.bin"):
    weights_path = "./model_fp32/pytorch_model.bin"
elif os.path.exists("./model_fp32/model.safetensors"):
    weights_path = "./model_fp32/model.safetensors"
else:
    raise FileNotFoundError("Could not find model weights file (.bin or .safetensors)")

file_size = os.path.getsize(weights_path) / 1024**2
print(f"✅ Loaded & Saved to './model_fp32' (Size: {file_size:.1f} MB)")

⬇️  Downloading distilbert-base-uncased-finetuned-sst-2-english...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Loaded & Saved to './model_fp32' (Size: 255.4 MB)


In [5]:
# Benchmark Function
def benchmark_model(model, text, steps=50):
    inputs = tokenizer(text, return_tensors="pt")
    
    # Warmup
    for _ in range(5):
        _ = model(**inputs)
        
    # Timing
    start = time.time()
    for _ in range(steps):
        with torch.no_grad():
            _ = model(**inputs)
    end = time.time()
    
    avg_time = (end - start) / steps * 1000
    return avg_time

sample_text = "Saturn Cloud makes scaling machine learning workloads incredibly easy and efficient."
time_fp32 = benchmark_model(model_fp32, sample_text)
print(f"⏱️ Standard (FP32) Latency: {time_fp32:.2f} ms")

⏱️ Standard (FP32) Latency: 48.62 ms


## 2. Dynamic Quantization (INT8)
We use `torch.quantization.quantize_dynamic` to convert the Linear layers to 8-bit integers. This requires **no retraining**.

In [6]:
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32, 
    {torch.nn.Linear},  # We only quantize the heavy Linear layers
    dtype=torch.qint8
)

# Verify size reduction
torch.save(model_int8.state_dict(), "quantized_model.pt")
size_int8 = os.path.getsize("quantized_model.pt") / 1024**2
print(f"📉 Quantized Model Size: {size_int8:.1f} MB")

/tmp/ipykernel_548476/1943256708.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


📉 Quantized Model Size: 132.3 MB


In [7]:
# Benchmark Quantized Model
time_int8 = benchmark_model(model_int8, sample_text)
print(f"⏱️ Quantized (INT8) Latency: {time_int8:.2f} ms")

speedup = time_fp32 / time_int8
print(f"🚀 Speedup: {speedup:.2f}x faster")

⏱️ Quantized (INT8) Latency: 33.94 ms
🚀 Speedup: 1.43x faster


## 3. NER Task (Token Classification)
Switching tasks is as easy as changing the pipeline model. We use `dslim/bert-base-NER` (or a smaller DistilBERT variant if available) for Named Entity Recognition.

In [8]:
ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

text = "Apple Inc. is planning to open a new store in Abuja, Nigeria next month."
entities = ner_pipe(text)

pd.DataFrame(entities)[['word', 'entity_group', 'score']]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

,word,entity_group,score
0,Apple Inc,ORG,0.999508
1,Abuja,LOC,0.998583
2,Nigeria,LOC,0.999648
